In [ ]:
import os
import sys
from pathlib import Path

repo_root = Path.cwd()
if repo_root.name == "scrapers":
    repo_root = repo_root.parent

if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))

from sofascore_scraper import SofascoreScraper


In [ ]:
import sys
from postgres_conn import PostgresDB

db = PostgresDB(
    db_url=os.getenv("DB_URL")
)
db.connect()

scraper = SofascoreScraper()

LEAGUES = {
    18653: "WE-League",
    214: "Damallsvenskan",
    10257: "Brasileirão Série A1",
    1894: "A-League Women",
    201: "Toppserien Women",
    10527: "Campeonato Nacional Feminino",
}
try:
    for league_id, league_name in LEAGUES.items():
        print(f"Scraping league: {league_name}")
        try:
            league_seasons = scraper.get_league_seasons(league_id)
            print(league_id, ", ", league_seasons)
            season_id = league_seasons["seasons"][0]["id"]
            raw_league_teams = scraper.get_league_season_standings(league_id, season_id)

            standings = raw_league_teams["standings"][0]["rows"]
            teams = [team["team"]["id"] for team in standings]

        except Exception as e:
            print(e)
            print(f"Failed to get teams for league {league_name}: {e}")
            continue
        print(teams)

        for team in teams:
            try:
                team_players = scraper.get_team_players(team)
                players = team_players["players"]
                for player in players:
                    player_id = player["player"]["id"]
                    print("Player: ", player_id)
                    try:
                        player_overview = scraper.get_player_bio(player_id)
                        player_stats = scraper.get_season_stats(player_id)
                    except Exception as inner_exc:
                        print(f"Skipping player {player_id}: {inner_exc}")
                        continue

                    db.save_raw_sofascore_json(player_id, player_overview)
                    db.save_raw_sofascore_json_season_stats(player_id, player_stats)
            except Exception as e:
                print(f"Failed to get players for team: {team}: {e}")
                continue
finally:
    try:
        db.close()
    except Exception:
        pass
